# DeepSea-20: Deep Reinforcement Learning with a Single Neural Network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Paper:** *Dirichlet Process Bayesian Neural Networks for Epistemic Uncertainty and Deep Exploration* (TMLR 2026)  
**Author:** Sumit Vashishtha  

---

### 1. Pedagogical Overview: The Deep Exploration Challenge
In reinforcement learning, exploration is often conflated with local dithering (such as $\epsilon$-greedy action selection). However, on sparse-reward, delayed-horizon tasks, local dithering requires an **exponential sample complexity** $\Omega(2^N)$ to stumble upon the goal by chance:
* For $N=20$: $2^{20} \approx 10^6$ episodes.
* For $N=50$: $2^{50} \approx 10^{15}$ episodes.

The canonical litmus test for deep exploration is the **DeepSea** environment (Osband et al., 2019):
* An $N \times N$ grid with episode horizon $H = N$.
* The agent starts at $(0, 0)$ and must navigate to $(N-1, N-1)$ to find the solitary terminal reward ($r = +1.0$).
* **Bottleneck 1:** Optimal actions are scrambled across states.
* **Bottleneck 2:** Traversing the optimal diagonal incurs a step penalty of $-0.01/N$ at every step.
* **Bottleneck 3:** A single mistake traps the agent in zero-reward states for the rest of the episode.

### 2. The Multi-Network Dilemma vs. Single-Network DP-DQN
Prior methods (such as Bootstrapped DQN with randomized priors, BootDQN-RP) solve DeepSea by instantiating an ensemble of $K = 20$ neural networks plus $K = 20$ randomized prior networks, scaling memory and compute by $\mathcal{O}(K)$ ($40\times$ parameter memory).

**DP-DQN** solves DeepSea using a **single neural network**:
1. At the start of each episode, the agent draws a random measure $\mathcal{P} \sim \mathrm{DP}$ using the constructive stick-breaking representation (**Vashishtha & Maillard, 2025**).
2. The single network is updated for a few warmstart steps on this composite measure.
3. The network parameters $\theta_k$ are frozen during the rollout, executing faithful **Thompson sampling** across the full horizon with the $\mathcal{O}(1)$ latency and memory of standard DQN!


In [ ]:
import os
import sys
from pathlib import Path
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Ensure repository root is on sys.path whether executed from root or notebooks/
notebook_dir = Path.cwd().resolve()
repo_root = str(notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import time
import numpy as np
import torch
import matplotlib.pyplot as plt

# Import tested DP-DQN engine from repository
from src.dp_dqn.config import DPDQNConfig
from src.dp_dqn.environments import make_env
from src.dp_dqn.base_measures import get_base_measure
from src.dp_dqn.agent import DPDQNAgent

print("DP-DQN engine successfully loaded.")


### 3. Setting Up DeepSea-20 with a Single MLP-20 Network
We configure DP-DQN with:
* Single MLP with LayerNorm (matching Osband et al., 2018).
* Model-free Maximum-Entropy Uniform optimistic base measure: $F_{0r} = \mathcal{U}(0, 1.0)$.
* Prior concentration $\alpha = 3.0$ and truncation $T = 50$ (99.9% expected residual mass rule).


In [ ]:
SIZE = 20
MAX_EPISODES = 600
SEED = 42

cfg = DPDQNConfig(
    env_name="deep_sea",
    deep_sea_size=SIZE,
    state_dim=SIZE * SIZE,
    action_dim=2,
    seed=SEED,
    alpha=3.0,
    batch_size=64,
    candidate_batch_size=96,
    base_measure="deep_sea_nondag_maxent",
    prior_reward_mean=0.5,
    prior_reward_std=0.288,
    hidden_dim=64,          # 2-layer MLP architecture
    num_layers=2,
    use_layer_norm=True,     # Strict LayerNorm enabled
    activation="relu",
    lr=1e-3,
    gamma=0.99,
    tau=0.05,
    sgd_period=2,
    buffer_capacity=1000000,
    target_warmstart=True,
    warmstart_steps=2,
    warmstart_lr_scale=0.8,
    num_episodes=MAX_EPISODES,
    max_episode_steps=SIZE,
    sampler_type="vashishtha_maillard",  # Vashishtha & Maillard stick-breaking sampler
    vm_prior_multiplier=10.0,
    one_living_network=True,             # Single neural network!
    use_td_info_gain_decay=True,
    td_info_scale=1.0,
    verbose=False,
)

env = make_env("deep_sea", size=SIZE, seed=SEED)
base_measure = get_base_measure("deep_sea_nondag_maxent", SIZE * SIZE, 2, deep_sea_size=SIZE)
agent = DPDQNAgent(cfg, base_measure=base_measure)

print(f"DeepSea-{SIZE} initialized with Single MLP (hidden_dim={cfg.hidden_dim}) Network!")


### 4. Running DeepSea-20: Pure Thompson Sampling in Action
Watch as DP-DQN discovers the terminal reward ($r=1.0$ at $(19, 19)$) and locks into the optimal policy, plateauing cumulative regret in only a few hundred episodes!


In [ ]:
returns = []
cum_regrets = []
cum_reg = 0.0
first_discovery = None
solved_episode = None

t0 = time.time()
print(f"--- Starting DP-DQN training on DeepSea-{SIZE} ---")

r_star = 1.0 - (1.0 / SIZE)

for ep in range(1, MAX_EPISODES + 1):
    agent.reset_episode()
    state = env.reset()
    ep_reward = 0.0
    done = False
    
    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.step(state, action, reward, next_state, done)
        state = next_state
        ep_reward += reward
            
    returns.append(ep_reward)
    
    regret = r_star - ep_reward  # Optimal return is 1.0 - 20*(0.01/20) = 0.99
    cum_reg += regret
    cum_regrets.append(cum_reg)
    
    if ep_reward > 0.5 and first_discovery is None:
        first_discovery = ep
        print(f"*** GOAL FIRST DISCOVERED at Episode {ep}! (Elapsed: {time.time()-t0:.1f}s) ***")
        
    if ep >= 100 and solved_episode is None:
        recent_mean = np.mean(returns[-50:])
        if recent_mean >= 0.90:
            solved_episode = ep
            print(f"*** ENVIRONMENT SOLVED at Episode {ep}! (50-ep mean return = {recent_mean:.2f}) ***")
            
    if ep % 50 == 0:
        recent_ret = np.mean(returns[-50:]) if len(returns) >= 50 else np.mean(returns)
        print(f"Episode {ep:3d}/{MAX_EPISODES} | Mean Return(50): {recent_ret:+.3f} | CumRegret: {cum_reg:.1f}")

total_time = time.time() - t0
print(f"Training completed in {total_time:.1f} seconds!")


### 5. Visualizing Cumulative Regret Plateau
In DeepSea, an agent that fails to explore suffers linear regret ($>9{,}900$, as vanilla DQN and BDQN do).  
DP-DQN reaches a cumulative regret plateau once the task is solved, exhibiting exploratory behavior similar to the findings reported in the paper.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cum_regrets, color="tab:blue", linewidth=2.5, label="DP-DQN (Single MLP-20)")
if first_discovery:
    plt.axvline(first_discovery, color="orange", linestyle="--", linewidth=1.5, label=f"First Discovery (Ep {first_discovery})")
if solved_episode:
    plt.axvline(solved_episode, color="green", linestyle=":", linewidth=2, label=f"Solved Plateau (Ep {solved_episode})")

plt.xlabel("Episodes", fontsize=12)
plt.ylabel("Cumulative Regret", fontsize=12)
plt.title(f"DP-DQN on DeepSea-{SIZE}: Sub-Linear Cumulative Regret Plateau", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 6. Pedagogical Takeaways
1. **Bypassing the $\Omega(2^N)$ Dithering Wall:** $\epsilon$-greedy agents have probability $(1/2)^{20} \approx 10^{-6}$ of reaching the goal by chance. DP-DQN's temporally consistent Thompson sampling follows coherent directional hypotheses, discovering the goal in ~100-300 episodes.
2. **Single-Network Bayesian Efficiency:** Unlike BootDQN-RP which requires maintaining 20 active networks and 20 prior networks ($40\times$ parameter memory and $20\times$ slower wall-clock time), DP-DQN achieves pure Thompson sampling with a **single neural network**.
3. **Reference:** For the theoretical foundation of Dirichlet Process posterior measure construction, see:
   * **Vashishtha, S., & Maillard, O.-A. (2025).** *Leveraging priors on distribution functions for multi-arm bandits.* Reinforcement Learning Journal, 6:1600–1623.
